In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

## **Access Folder and Create Dataframe**

For activity level distribution, dormant customers (inactive customers) defined by assuming at least these customer have one transaction each month.

In [ ]:
# Sample Configuration

N_CUSTOMERS = 400
SAMPLING_SEED = 42

# Original source-data folder:
# contains customers.csv and transactions_2024.csv
folder_path = (r"data_source/")

dataframes = {}

for file in os.listdir(folder_path):
    if file.lower().endswith(".csv"):
        df_name = os.path.splitext(file)[0]
        full_path = os.path.join(folder_path, file)

        dataframes[df_name] = pd.read_csv(full_path)

In [ ]:
# load transaction and customer data

transactions_df = dataframes["transactions_2024"].copy() #to duplicate original customer data
customers_df = dataframes["customers"].copy() #to duplicate original customer data


print("Source data loaded successfully.")

In [ ]:
print(f"\n\033[1m ~~~~~ FULL TRANSACTION DATA SANITY CHECK ~~~~~ \033[0m\n")

# Dimensions & Info
print("Dimension consistency check:")
print(f"  Actual transactions      : {len(transactions_df):,}")
print(f"Dimension transactions    : {transactions_df.shape}")
print(f"Columns: {transactions_df.columns.tolist()}")

print("\n\033[1m--- DATA INFO & TYPES ---\n\033[0m")
transactions_df.info()
print()

# Missing Values
print(f'\033[1m----- MISSING VALUES -----\033[0m\n{transactions_df.isnull().sum()}\n')
# Duplicates
print(f'\033[1m----- DUPLICATE CLIENT_ID -----\033[0m\n{transactions_df.duplicated(subset=["client_id"]).sum()}\n')
# Statistical Summary
print("\033[1m----- STATISTICAL DESCRIPTION -----\033[0m")
display(transactions_df.describe(include="all").T)


In [ ]:
# Display dataset
print("\n\033[1m----- TRANSACTION DATASET -----\n\033[0m")
display(transactions_df.head())

In [ ]:
print(f"\n\033[1m ~~~~~ CUSTOMER DATA SANITY CHECK ~~~~~ \033[0m\n")

# Dimensions & Info
print("Dimension consistency check:")
print(f'  Actual customers      : {len(customers_df):,}')
print(f'Dimension customers    : {customers_df.shape}')
print(f'Columns: {customers_df.columns.tolist()}')


print("\n\033[1m----- DATA INFO & TYPES -----\033[0m")
customers_df.info()
print()

# Missing Values
print(f'\033[1m----- MISSING VALUES -----\033[0m\n{customers_df.isnull().sum()}\n')

# Duplicates by client_id
print(f'\033[1m----- DUPLICATE CLIENT_ID -----\033[0m\n{customers_df.duplicated(subset=["client_id"]).sum()}\n')


# Duplicates by transaction_id 
#print(f'\033[1m----- DUPLICATE CLIENT_ID -----\033[0m\n{customers_df.duplicated(subset=["transaction_id"]).sum()}\n')

# Statistical Summary
print("\033[1m----- STATISTICAL DESCRIPTION -----\033[0m")
display(customers_df.describe(include="all").T)


In [ ]:
# Every transaction should belong to a customer in customer master
transaction_clients = set(transactions_df["client_id"].unique())
customer_clients = set(customers_df["client_id"].unique())

missing_customer_ids = transaction_clients - customer_clients

print(
    f"\nTransaction client IDs not found in customer master: "
    f"{len(missing_customer_ids):,}"
)

assert len(missing_customer_ids) == 0, (
    "Some transaction client IDs are absent from customer master data."
)

print("Customer–transaction relationship check: PASSED")

In [ ]:
# Display dataset
print("\n\033[1m----- CUSTOMER DATASET -----\n\033[0m")
display(customers_df.head())


## **Stratified Sampling: Function**

In [ ]:
def sample_customers_with_full_trx(
    customers_df,
    transactions_df,
    n_customers=400,
    random_state=42 # To ensure reproducibility and consistent sampling for audit trails
):
    # 1. Compute txn count per customer
    txn_counts = (
        transactions_df
        .groupby("client_id")
        .size()
        .reset_index(name="txn_count")
    )

    # 2. Assign activity level
    txn_counts["activity_level"] = pd.cut(
        txn_counts["txn_count"],
        bins=[0, 12, 50, 200, 9999],
        labels=["Dormant", "Low", "Moderate", "Heavy"],
        include_lowest=True
    )

    # 3. Stratified sample proportional to population
    sampled = (
        txn_counts
        .groupby("activity_level", group_keys=False)
        .apply(lambda x: x.sample(
            n=max(1, round(len(x) * n_customers / len(txn_counts))), #ensure that there is at least 1 person representing each group
            random_state=random_state
        ))
    )
    sampled_ids = sampled["client_id"].values

    # 4: All transactions for sampled customers
    transactions_out = transactions_df[
        transactions_df["client_id"].isin(sampled_ids)
    ].copy()

    # 5. customer master data for sampled customers
    customers_out = customers_df[
        customers_df["client_id"].isin(sampled_ids)
    ].copy()

    # Validation
    print("\n\033[1m~~~~~CUSTOMER-COMPLETE SAMPLING RESULT~~~~~\n\033[0m")
    print(f"Target customers     : {n_customers}")
    print(f"Actual customers     : {len(sampled_ids):,}")
    print(f"Total transactions   : {len(transactions_out):,}")
    print(f"Avg txn/customer     : {len(transactions_out)/len(sampled_ids):.1f}")
    print(f"Population avg       : {len(transactions_df)/len(txn_counts):.1f}")
    print()

    print("\n\033[1m~~~~~Activity level distribution (sample vs population):~~~~~\n\033[0m")
    s_dist = sampled["activity_level"].value_counts(normalize=True).round(3)
    p_dist = txn_counts["activity_level"].value_counts(normalize=True).round(3)
    cmp    = pd.DataFrame({"Sample": s_dist, "Population": p_dist})
    cmp["Diff"] = (cmp["Sample"] - cmp["Population"]).abs()
    print(cmp)
    print()

    # Monthly coverage
    transactions_out["date"]  = pd.to_datetime(transactions_out["date"])
    transactions_out["month"] = transactions_out["date"].dt.to_period("M")
    monthly = transactions_out.groupby("month").size()
    print("\n\033[1m~~~~~Monthly transaction coverage:~~~~~\n\033[0m")
    print(monthly.to_string())
    print(f"\n\033[1mMonths with 0 transactions: {(monthly == 0).sum()} <- must be 0\033[0m")
    transactions_out.drop(columns=["month"], inplace=True)

    return customers_out, transactions_out


In [ ]:
# Run customer-complete sampling
customers_sampled, transactions_sampled = sample_customers_with_full_trx(
    customers_df,
    transactions_df,
    n_customers= 400,
    random_state= 42
)


In [ ]:
print("\n\033[1m-----SAMPLE DATASET RESULT-----\n\033[0m")
print(f"customers_sampled    : {customers_sampled.shape}")
print(f"transactions_sampled : {transactions_sampled.shape}")

In [ ]:
print("\n\033[1m 1. customers_sampled:\n\033[0m")
display(customers_sampled.head())

print("\n\033[1m 2. transactions_sampled:\n\033[0m")
display(transactions_sampled.head())

In [ ]:
# Save sampled files — run ONCE, reload on subsequent runs
cust_sampled_path = os.path.join(folder_path, "customers_sampled_400.csv")
trx_sampled_path  = os.path.join(folder_path, "transactions_sampled_400.csv")

if os.path.exists(trx_sampled_path) and os.path.exists(cust_sampled_path):
    print("Sampled files already exist — loading...")
    customers_sampled    = pd.read_csv(cust_sampled_path)
    transactions_sampled = pd.read_csv(trx_sampled_path)
    print(f"customers_sampled    : {customers_sampled.shape}")
    print(f"transactions_sampled : {transactions_sampled.shape}")
else:
    customers_sampled, transactions_sampled = sample_customers_with_full_trx(
        customers_df,
        transactions_df,
        n_customers = 400,
        random_state = 42
    )
    customers_sampled.to_csv(cust_sampled_path, index=False)
    transactions_sampled.to_csv(trx_sampled_path, index=False)
    print(f"\n Saved: customers_sampled_400.csv    {customers_sampled.shape}")
    print(f" Saved: transactions_sampled_400.csv {transactions_sampled.shape}")

print(f"\n\033[1m -- Used by: AS-IS monitoring pipeline + SDG pipeline + TO-BE simulation --\033[0m")
